# NB05 — PTSAM Soft-Prompt Tuning on Potsdam

**Runs on Kaggle T4 GPU.**

**Datasets to attach:**
- `dummyirl/sam3-weights` — SAM3 checkpoint
- `dummyirl/6isprs` — Potsdam GeoTIFF + labels

**What this does:**
1. Freeze ALL of SAM3 (PE-L+ encoder + text encoder + mask decoder weights)
2. Create 8 learnable soft tokens (8×256 = 2048 trainable params total)
3. Inject soft tokens into `language_features` before each mask-decoder call
4. Train on 5 Potsdam tiles (all except 6_15 val) with resolution bridge aug
5. Evaluate mIoU on tile 6_15
6. Save `soft_prompts.pt` → load in NB02 for improved inference

**Why only 2048 params won't overfit on 5 images:**
A single transformer attention head has ~1.8M params. PTSAM uses 2048 total —
there simply isn't capacity to memorize tile-specific details. The optimization
finds generalizable "aerial imagery style" signal instead.
Reference: PTSAM (CVPRW 2025).

## 1 — Environment setup

In [ ]:
import os

!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda_installer.sh
!bash /tmp/miniconda_installer.sh -b -p /tmp/miniconda

os.environ.pop('PYTHONPATH', None)
os.environ['PATH'] = '/tmp/miniconda/bin:' + os.environ['PATH']

!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
!conda --version


In [ ]:
!/tmp/miniconda/bin/conda create -n segearth python=3.10 -y


In [ ]:
!conda run -n segearth pip install torch==2.4.0 torchvision==0.19.0 -q


In [ ]:
!conda run -n segearth pip install openmim -q
!conda run -n segearth mim install 'mmcv==2.2.0' -q
!conda run -n segearth pip install 'mmsegmentation==1.2.2' -q


In [ ]:
%%bash
source /tmp/miniconda/bin/activate segearth
python - << 'EOF'
import pathlib
f = pathlib.Path('/tmp/miniconda/envs/segearth/lib/python3.10/site-packages/mmseg/__init__.py')
f.write_text(f.read_text().replace("MMCV_MAX = '2.2.0'", "MMCV_MAX = '2.3.0'"))
print('Patched MMCV_MAX -> 2.3.0')
EOF
pip install numpy==1.26.4 tifffile -q


In [ ]:
%%bash
source /tmp/miniconda/bin/activate segearth
python - << 'EOF'
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0))
import tifffile; print('tifffile OK')
EOF


## 2 — Clone our fork

In [ ]:
import subprocess, os
from pathlib import Path

REPO = Path('/tmp/SegEarth-OV-3')

if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
    print(f'Updated -> {REPO}')
else:
    subprocess.run(
        ['git', 'clone', '--depth=1',
         'https://github.com/HarishDeepak/rg-segearth-ov3', str(REPO)],
        check=True)
    print(f'Cloned -> {REPO}')

os.chdir(REPO)
!conda run -n segearth pip install -r requirements.txt -q


## 3 — PTSAM Training

**Config:**
- Soft tokens: 8 × 256 = 2048 trainable params
- Training tiles: 5_14, 5_15, 6_13, 6_14, 7_13  (val = 6_15)
- Patches per tile: 10 random 512×512 crops
- Resolution bridge: 30% chance downsample ×4 then upsample (simulates 20cm GSD)
- Epochs: 50 (adjust with N_EPOCHS below)
- Optimizer: Adam lr=1e-3 with cosine decay
- Loss: cross-entropy, ignore_index=255

In [ ]:
%%bash
export MPLBACKEND=Agg
export PYTHONUNBUFFERED=1
source /tmp/miniconda/bin/activate segearth
cd /tmp/SegEarth-OV-3

python - << 'PYEOF'
import sys, torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, tifffile, random, time, math
from pathlib import Path

sys.stdout.reconfigure(line_buffering=True)

torch.manual_seed(42); np.random.seed(42); random.seed(42)
torch.backends.cudnn.benchmark = True

# ── CONFIG ────────────────────────────────────────────────────────────────────
N_SOFT     = 8
DIM        = 256
LR         = 1e-3
PATCHES_PER_TILE = 10
CROP_SIZE  = 512
RES_BRIDGE_PROB = 0.30
DEVICE     = "cuda"

# ── SESSION CONFIG — edit these between Kaggle sessions ──────────────────────
TOTAL_EPOCHS = 50
START_EPOCH  = 1
N_EPOCHS     = 20
RESUME_PATH  = None
# ─────────────────────────────────────────────────────────────────────────────

POTSDAM_ROOT  = Path("/kaggle/input/datasets/dummyirl/6isprs")
TRAIN_TILES   = ["5_14", "5_15", "6_13", "6_14", "7_13"]
N_CLASSES     = 6
IGNORE_IDX    = 255
SAVE_PATH     = Path("/kaggle/working/soft_prompts.pt")
END_EPOCH     = START_EPOCH + N_EPOCHS - 1

TRAIN_CLASS_NAMES = [
    "impervious surface, road, pavement, paved ground",
    "building, rooftop, structure",
    "low vegetation, grass, shrub, lawn, meadow",
    "tree, forest, canopy",
    "car, vehicle",
    "clutter, background",
]
RGB_TO_IDX = {
    (255, 255, 255): 1, (  0,   0, 255): 2, (  0, 255, 255): 3,
    (  0, 255,   0): 4, (255, 255,   0): 5, (255,   0,   0): 6,
}

print(f"Session: epochs {START_EPOCH}-{END_EPOCH} of {TOTAL_EPOCHS} total", flush=True)

def rgb_to_gt(label_path):
    arr = tifffile.imread(str(label_path))
    if arr.ndim == 2:
        return None
    h, w = arr.shape[:2]
    gt = torch.full((h, w), IGNORE_IDX, dtype=torch.int64)
    for (r, g, b), idx in RGB_TO_IDX.items():
        mask = (arr[:,:,0]==r) & (arr[:,:,1]==g) & (arr[:,:,2]==b)
        gt[mask] = idx - 1
    return gt

def resolution_bridge(img_t):
    s = img_t.shape[-2:]
    down = F.interpolate(img_t, scale_factor=0.25, mode="bilinear", align_corners=False)
    return F.interpolate(down, size=s, mode="bilinear", align_corners=False)

def bb_to_cpu(bb_out):
    def _cpu(v):
        if isinstance(v, torch.Tensor): return v.detach().cpu()
        if isinstance(v, (list, tuple)): return type(v)(_cpu(x) for x in v)
        return v
    return {k: _cpu(v) for k, v in bb_out.items()}

def bb_to_gpu(bb_cpu, device):
    def _gpu(v):
        if isinstance(v, torch.Tensor): return v.to(device, non_blocking=True)
        if isinstance(v, (list, tuple)): return type(v)(_gpu(x) for x in v)
        return v
    return {k: _gpu(v) for k, v in bb_cpu.items()}

from config_local import SAM3_CHECKPOINT
from sam3 import build_sam3_image_model
from sam3.model.data_misc import FindStage

print("Loading SAM3...", flush=True)
model = build_sam3_image_model(
    bpe_path="./sam3/assets/bpe_simple_vocab_16e6.txt.gz",
    checkpoint_path=SAM3_CHECKPOINT, device=DEVICE,
)
model.eval()
for param in model.parameters():
    param.requires_grad = False
print(f"Frozen {sum(p.numel() for p in model.parameters()):,} params.", flush=True)

# ── AMP: BF16 on Ampere+ (cc>=8.0), FP16+GradScaler on T4/Turing (cc<8.0) ──
_cc_major = torch.cuda.get_device_capability()[0]
_bf16_ok  = _cc_major >= 8
AMP_DTYPE = torch.bfloat16 if _bf16_ok else torch.float16
scaler    = None if _bf16_ok else torch.amp.GradScaler("cuda")
print(f"GPU: {torch.cuda.get_device_name(0)}  cc={torch.cuda.get_device_capability()}", flush=True)
print(f"AMP: {AMP_DTYPE}  scaler: {'none (BF16)' if scaler is None else 'GradScaler (FP16)'}", flush=True)

soft_prompts = nn.Parameter(torch.randn(N_SOFT, 1, DIM, device=DEVICE) * 0.02)
optimizer    = torch.optim.Adam([soft_prompts], lr=LR)
scheduler    = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=TOTAL_EPOCHS, eta_min=1e-5
)

if RESUME_PATH is not None and START_EPOCH > 1:
    rp = Path(RESUME_PATH)
    if not rp.exists():
        raise FileNotFoundError(f"RESUME_PATH not found: {rp}")
    soft_prompts.data = torch.load(str(rp), map_location=DEVICE)
    for _ in range(START_EPOCH - 1):
        scheduler.step()
    print(f"Resumed {rp.name}  LR={scheduler.get_last_lr()[0]:.2e}", flush=True)
elif START_EPOCH > 1:
    print(f"WARNING: START_EPOCH={START_EPOCH} but RESUME_PATH=None", flush=True)

find_stage = FindStage(
    img_ids=torch.tensor([0], device=DEVICE, dtype=torch.long),
    text_ids=torch.tensor([0], device=DEVICE, dtype=torch.long),
    input_boxes=None, input_boxes_mask=None, input_boxes_label=None,
    input_points=None, input_points_mask=None,
)

print("Precomputing text embeddings...", flush=True)
text_embeds = []
with torch.no_grad():
    for cls_name in TRAIN_CLASS_NAMES:
        te = model.backbone.forward_text([cls_name], device=DEVICE)
        text_embeds.append({k: v for k, v in te.items()})
print(f"{N_CLASSES} embeddings cached.", flush=True)

from torchvision.transforms import v2
img_transform = v2.Compose([
    v2.ToDtype(torch.uint8, scale=True),
    v2.Resize(size=(1008, 1008)),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

print(f"Loading tiles from {POTSDAM_ROOT}  exists={POTSDAM_ROOT.exists()}", flush=True)
all_patches, all_backbone = [], []

for tile in TRAIN_TILES:
    img_path = POTSDAM_ROOT / f"top_potsdam_{tile}_RGB.tif"
    lbl_path = POTSDAM_ROOT / f"top_potsdam_{tile}_label_noBoundary.tif"
    if not img_path.exists():
        print(f"  SKIP {tile}", flush=True); continue
    img_arr = tifffile.imread(str(img_path))[:, :, :3]
    gt_full  = rgb_to_gt(lbl_path)
    if gt_full is None:
        print(f"  SKIP {tile} (label format)", flush=True); continue
    h, w = img_arr.shape[:2]
    n, attempts = 0, 0
    while n < PATCHES_PER_TILE and attempts < PATCHES_PER_TILE * 5:
        attempts += 1
        y = random.randint(0, max(0, h - CROP_SIZE))
        x = random.randint(0, max(0, w - CROP_SIZE))
        gt_crop = gt_full[y:y+CROP_SIZE, x:x+CROP_SIZE]
        if (gt_crop != IGNORE_IDX).sum() < CROP_SIZE * CROP_SIZE * 0.2: continue
        img_t = torch.from_numpy(img_arr[y:y+CROP_SIZE, x:x+CROP_SIZE]).permute(2,0,1).float()/255.0
        all_patches.append((img_t, gt_crop.to(DEVICE)))
        n += 1
    print(f"  {tile}: {n} patches", flush=True)

if not all_patches:
    raise RuntimeError("No patches found. Check dummyirl/6isprs is attached.")

print(f"Caching {len(all_patches)} backbone features on CPU...", flush=True)
for i, (img_t, _) in enumerate(all_patches):
    with torch.no_grad(), torch.amp.autocast("cuda", dtype=AMP_DTYPE):
        bb_out = model.backbone.forward_image(img_transform(img_t.unsqueeze(0).to(DEVICE)))
    all_backbone.append(bb_to_cpu(bb_out))
    if (i+1) % 10 == 0:
        print(f"  {i+1}/{len(all_patches)} cached", flush=True)
torch.cuda.empty_cache()
print(f"Cache done. Starting training epochs {START_EPOCH}-{END_EPOCH}...", flush=True)

best_loss = float("inf")
t0 = time.time()

for epoch in range(START_EPOCH, END_EPOCH + 1):
    epoch_loss = 0.0
    for patch_idx in random.sample(range(len(all_patches)), len(all_patches)):
        img_t, gt_t = all_patches[patch_idx]

        if random.random() < RES_BRIDGE_PROB:
            with torch.no_grad(), torch.amp.autocast("cuda", dtype=AMP_DTYPE):
                aug_img = resolution_bridge(img_t.unsqueeze(0).to(DEVICE))
                aug_bb  = model.backbone.forward_image(img_transform(aug_img))
            cur_bb = aug_bb
        else:
            cur_bb = bb_to_gpu(all_backbone[patch_idx], DEVICE)

        H, W = gt_t.shape
        all_logits = []

        with torch.amp.autocast("cuda", dtype=AMP_DTYPE):
            for cls_idx in range(N_CLASSES):
                dummy_geom = model._get_dummy_prompt()   # fresh per class — forward_grounding mutates it
                te = text_embeds[cls_idx]
                lang_feat = te["language_features"].detach()
                lang_mask = te["language_mask"].detach()
                soft      = soft_prompts.expand(-1, 1, -1)
                soft_mask = torch.ones(1, N_SOFT, device=DEVICE, dtype=lang_mask.dtype)
                this_bb   = dict(cur_bb)
                this_bb["language_features"] = torch.cat([lang_feat, soft], dim=0)
                this_bb["language_mask"]     = torch.cat([lang_mask, soft_mask], dim=1)
                outputs = model.forward_grounding(
                    backbone_out=this_bb, find_input=find_stage,
                    geometric_prompt=dummy_geom, find_target=None,
                )
                sem  = F.interpolate(outputs["semantic_seg"], size=(H,W),
                                     mode="bilinear", align_corners=False)
                pres = outputs["presence_logit_dec"].sigmoid()
                all_logits.append(sem.squeeze() * pres.squeeze())

            logits = torch.stack(all_logits, dim=0).unsqueeze(0)
            loss   = F.cross_entropy(logits, gt_t.unsqueeze(0), ignore_index=IGNORE_IDX)

        optimizer.zero_grad()
        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_([soft_prompts], max_norm=1.0)
            scaler.step(optimizer); scaler.update()
        else:
            loss.backward()
            nn.utils.clip_grad_norm_([soft_prompts], max_norm=1.0)
            optimizer.step()
        epoch_loss += loss.item()

    scheduler.step()
    avg_loss = epoch_loss / len(all_patches)
    elapsed  = time.time() - t0
    eta      = elapsed / (epoch - START_EPOCH + 1) * (END_EPOCH - epoch)
    print(f"Epoch {epoch:3d}/{TOTAL_EPOCHS}  loss={avg_loss:.4f}  "
          f"lr={scheduler.get_last_lr()[0]:.2e}  "
          f"elapsed={elapsed/60:.1f}m  ETA={eta/60:.1f}m", flush=True)

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(soft_prompts.detach().cpu(), str(SAVE_PATH))

    if epoch % 10 == 0:
        ckpt = Path(str(SAVE_PATH).replace('.pt', f'_ep{epoch:03d}.pt'))
        torch.save(soft_prompts.detach().cpu(), str(ckpt))
        print(f"  Checkpoint -> {ckpt.name}  "
              f"(next session: START_EPOCH={epoch+1}, RESUME_PATH='{ckpt.name}')", flush=True)

torch.save(soft_prompts.detach().cpu(), str(SAVE_PATH))
print(f"\nDone. Epochs {START_EPOCH}-{END_EPOCH}. Best loss={best_loss:.4f}.", flush=True)
print(f"Total time: {(time.time()-t0)/60:.1f} min", flush=True)
if END_EPOCH < TOTAL_EPOCHS:
    print(f"Next session: START_EPOCH={END_EPOCH+1}, RESUME_PATH='soft_prompts_ep{END_EPOCH:03d}.pt'", flush=True)
PYEOF


## 4 — Quick eval on val tile 6_15

Loads the saved `soft_prompts.pt` and computes mIoU on tile `6_15`
using the same sliding-window eval as NB02 (so results are directly comparable).

> Expected improvement over zero-shot baseline: +3–8% mIoU if training converged.
> If soft_prompts.pt not saved, this cell will error — run training cell first.

In [ ]:
%%bash
export MPLBACKEND=Agg
source /tmp/miniconda/bin/activate segearth
cd /tmp/SegEarth-OV-3

python - << 'PYEOF'
import torch, torch.nn.functional as F
import numpy as np, tifffile
from pathlib import Path
from PIL import Image

DEVICE       = "cuda"
SOFT_PATH    = Path("/kaggle/working/soft_prompts.pt")
POTSDAM_ROOT = Path("/kaggle/input/datasets/dummyirl/6isprs")
VAL_TILE     = "6_15"
N_CLASSES    = 6
IGNORE_IDX   = 255
CROP_SIZE    = 1024

if not SOFT_PATH.exists():
    print("soft_prompts.pt not found — run training cell first.")
    raise SystemExit(1)

soft_prompts = torch.nn.Parameter(torch.load(str(SOFT_PATH)).to(DEVICE))
print(f"Loaded soft_prompts {soft_prompts.shape}  ({soft_prompts.numel()} params)")

from config_local import SAM3_CHECKPOINT
from sam3 import build_sam3_image_model
from sam3.model.data_misc import FindStage

model = build_sam3_image_model(
    bpe_path="./sam3/assets/bpe_simple_vocab_16e6.txt.gz",
    checkpoint_path=SAM3_CHECKPOINT, device=DEVICE)
model.eval()
for p in model.parameters():
    p.requires_grad = False

find_stage = FindStage(
    img_ids=torch.tensor([0], device=DEVICE, dtype=torch.long),
    text_ids=torch.tensor([0], device=DEVICE, dtype=torch.long),
    input_boxes=None, input_boxes_mask=None, input_boxes_label=None,
    input_points=None, input_points_mask=None,
)

CLASS_NAMES = [
    "impervious surface, road, pavement, paved ground",
    "building, rooftop, structure",
    "low vegetation, grass, shrub, lawn, meadow",
    "tree, forest, canopy",
    "car, vehicle",
    "clutter, background",
]
RGB_TO_IDX = {
    (255,255,255):1, (0,0,255):2, (0,255,255):3,
    (0,255,0):4,     (255,255,0):5, (255,0,0):6
}

def rgb_to_gt(path):
    arr = tifffile.imread(str(path))
    if arr.ndim == 2: return None
    h, w = arr.shape[:2]
    gt = torch.full((h,w), IGNORE_IDX, dtype=torch.int64)
    for (r,g,b), idx in RGB_TO_IDX.items():
        m = (arr[:,:,0]==r)&(arr[:,:,1]==g)&(arr[:,:,2]==b)
        gt[m] = idx - 1
    return gt

from torchvision.transforms import v2
img_transform = v2.Compose([
    v2.ToDtype(torch.uint8, scale=True),
    v2.Resize(size=(1008,1008)),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5]),
])

img_path = POTSDAM_ROOT / f"top_potsdam_{VAL_TILE}_RGB.tif"
lbl_path = POTSDAM_ROOT / f"top_potsdam_{VAL_TILE}_label_noBoundary.tif"
img_arr = tifffile.imread(str(img_path))[:,:,:3]
gt_full  = rgb_to_gt(lbl_path)
H_full, W_full = img_arr.shape[:2]
print(f"Val tile {VAL_TILE}: {W_full}x{H_full}px,  valid GT pixels: {(gt_full!=IGNORE_IDX).sum()}")

# Text embeddings (frozen)
text_embeds = []
with torch.no_grad():
    for n in CLASS_NAMES:
        text_embeds.append(model.backbone.forward_text([n], device=DEVICE))

# Sliding-window prediction
pred_full = torch.zeros(N_CLASSES, H_full, W_full, device=DEVICE)
weight_mat = torch.zeros(H_full, W_full, device=DEVICE)
STRIDE = CROP_SIZE

import math
h_grids = math.ceil((H_full - CROP_SIZE) / STRIDE) + 1
w_grids = math.ceil((W_full - CROP_SIZE) / STRIDE) + 1
total = h_grids * w_grids
print(f"Sliding window: {h_grids}x{w_grids}={total} crops...")

for hi in range(h_grids):
    for wi in range(w_grids):
        y1 = min(hi * STRIDE, H_full - CROP_SIZE)
        x1 = min(wi * STRIDE, W_full - CROP_SIZE)
        y2, x2 = y1 + CROP_SIZE, x1 + CROP_SIZE

        crop = img_arr[y1:y2, x1:x2]
        img_t = torch.from_numpy(crop).permute(2,0,1).float().unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            bb = model.backbone.forward_image(img_transform(img_t))

        crop_logits = torch.zeros(N_CLASSES, CROP_SIZE, CROP_SIZE, device=DEVICE)

        with torch.no_grad():
            for cls_idx, te in enumerate(text_embeds):
                dummy_geom = model._get_dummy_prompt()   # fresh per class — forward_grounding mutates it
                lang_f = te["language_features"].detach()
                lang_m = te["language_mask"].detach()
                soft   = soft_prompts.expand(-1, 1, -1)
                soft_m = torch.ones(1, soft.shape[0], device=DEVICE, dtype=lang_m.dtype)
                this_bb = dict(bb)
                this_bb["language_features"] = torch.cat([lang_f, soft], dim=0)
                this_bb["language_mask"]     = torch.cat([lang_m, soft_m], dim=1)
                out = model.forward_grounding(this_bb, find_stage, dummy_geom, None)
                sem = F.interpolate(out["semantic_seg"], size=(CROP_SIZE,CROP_SIZE),
                                    mode="bilinear", align_corners=False)
                pres = out["presence_logit_dec"].sigmoid()
                crop_logits[cls_idx] = sem.squeeze() * pres.squeeze()

        pred_full[:, y1:y2, x1:x2] += crop_logits
        weight_mat[y1:y2, x1:x2]   += 1.0

pred_full = pred_full / weight_mat.unsqueeze(0)
seg_pred = pred_full.argmax(0).cpu()

# mIoU
tp = torch.zeros(N_CLASSES); fp = torch.zeros(N_CLASSES); fn = torch.zeros(N_CLASSES)
valid = gt_full != IGNORE_IDX
for c in range(N_CLASSES):
    pc = (seg_pred == c) & valid
    lc = (gt_full  == c) & valid
    tp[c] = (pc & lc).sum().float()
    fp[c] = (pc & ~lc).sum().float()
    fn[c] = (~pc & lc).sum().float()

iou = tp / (tp + fp + fn + 1e-6)
CLS_NAMES = ["impervious","building","low_veg","tree","car","clutter"]
print("\n=== Val tile 6_15 — PTSAM mIoU ===")
for i, (n, v) in enumerate(zip(CLS_NAMES, iou)):
    print(f"  {n:<20} {v*100:6.2f}%")
print(f"  {'mIoU':<20} {iou.mean()*100:6.2f}%")
PYEOF


## 5 — Download soft_prompts.pt

Run this cell to see the download link for `soft_prompts.pt`.
Upload it to a Kaggle dataset so NB02 can use it at inference time.

In [ ]:
from IPython.display import FileLink
FileLink('/kaggle/working/soft_prompts.pt')


In [ ]:
## 6 — Upload soft_prompts.pt to Hugging Face (auto-save, don't lose it)
#
# BEFORE RUNNING: Add your HF token as a Kaggle secret:
#   Notebook -> Settings -> Secrets -> Add new secret
#   Name: HF_TOKEN
#   Value: your HuggingFace write token (hf_...)
#
# This uploads to HarishDeepak/geo-prompt-peft-checkpoints under
# segearth-ov3/soft_prompts.pt so it survives Kaggle session expiry.

from pathlib import Path

SOFT_PATH = Path('/kaggle/working/soft_prompts.pt')
HF_REPO   = 'HarishDeepak/geo-prompt-peft-checkpoints'
HF_PATH   = 'segearth-ov3/soft_prompts.pt'

if not SOFT_PATH.exists():
    print('soft_prompts.pt not found -- run training cell first.')
else:
    !pip install huggingface_hub -q
    from huggingface_hub import HfApi
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        import os
        hf_token = os.environ.get('HF_TOKEN', '')
    if not hf_token:
        print('HF_TOKEN not found. Add it as a Kaggle secret (Settings -> Secrets).')
    else:
        api = HfApi()
        url = api.upload_file(
            path_or_fileobj=str(SOFT_PATH),
            path_in_repo=HF_PATH,
            repo_id=HF_REPO,
            repo_type='model',
            token=hf_token,
            commit_message='NB05 PTSAM soft_prompts.pt -- 2048 params on Potsdam',
        )
        print(f'Uploaded -> {url}')
        print(f'Download later: from huggingface_hub import hf_hub_download')
        print(f'  hf_hub_download(repo_id="{HF_REPO}", filename="{HF_PATH}", repo_type="model")')
